# HSI α-sweep + raw-score archival

Extends `hsi_joint_cv.ipynb` with two goals:

1. **α sweep**: report all four methods (D/A/B/C) at α ∈ {0.05, 0.10, 0.15}. CV for (r, bw) is still done at α=0.10 to avoid multiple-α hyperparameter tuning.
2. **Archive raw scores**: save softmax probs + raw APS scores + SACP-smoothed scores into each pkl so that future re-analyses (SSCV, λ/k sensitivity, new metrics) never need to retrain the 3D-CNN.

Output: `/content/drive/MyDrive/hsi_alpha_sweep/checkpoints/{ds}_seed{k}.pkl` — 50 enriched pkls.

**Expected runtime**: ~1h on a T4 (same as `hsi_joint_cv`, since the marginal cost of extra α thresholds is negligible). If GPU crashes, reruns resume from cached pkls.

## Resume note
Every pkl is atomically written (`.tmp` → `rename`). If the kernel dies mid-run, restart; only incomplete datasets rerun.


## 1. Setup + GPU check

In [1]:
!pip install -q scikit-learn scipy matplotlib

import os, sys, time, gc, pickle, json, glob, csv
import numpy as np
import scipy.io as sio
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


python 3.12.13 | torch 2.10.0+cu128 | cuda True
GPU: Tesla T4


## 2. Mount Drive + paths

In [2]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Reuse datasets from the original Drive workspace
SACP_GEOCP_DIR = '/content/drive/MyDrive/sacp_geocp'
OLD_DATA_DIR   = f'{SACP_GEOCP_DIR}/datasets'

# --- LOCAL workspace (primary): fast + immune to Drive FUSE disconnects ---
LOCAL_DIR       = '/content/hsi_alpha_sweep'
CKPT_DIR        = f'{LOCAL_DIR}/checkpoints'
RESULTS_DIR     = f'{LOCAL_DIR}/results'
FIG_DIR         = f'{LOCAL_DIR}/figures'

# --- DRIVE mirror (backup): copied to after each seed, best-effort ---
DRIVE_MIRROR    = '/content/drive/MyDrive/hsi_alpha_sweep'
DRIVE_CKPT_DIR  = f'{DRIVE_MIRROR}/checkpoints'

DATA_DIR = OLD_DATA_DIR
for d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Restore any cached pkls from Drive to local on cold start (so resume still works
# even if the runtime was recycled and local storage was wiped).
import shutil
def _safe_mkdir(d):
    try: os.makedirs(d, exist_ok=True); return True
    except OSError: return False

def _restore_from_drive():
    if not _safe_mkdir(DRIVE_CKPT_DIR):
        print('[restore] Drive not ready — skipping (will run everything fresh)'); return 0
    try:
        pkls = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pkl')]
    except OSError as e:
        print(f'[restore] list failed: {e}'); return 0
    restored = 0
    for f in pkls:
        src = f'{DRIVE_CKPT_DIR}/{f}'; dst = f'{CKPT_DIR}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(src, dst); restored += 1
        except OSError as e: print(f'[restore] {f} failed: {e}')
    return restored
n_restored = _restore_from_drive()
print(f'[restore] copied {n_restored} cached pkls from Drive to local')

print(f'DATA_DIR : {DATA_DIR}')
print(f'CKPT_DIR : {CKPT_DIR}  (local, primary)')
print(f'MIRROR   : {DRIVE_CKPT_DIR}  (Drive, best-effort backup)')

expected_mats = [('ip',  'Indian_pines_corrected.mat'),
                 ('pu',  'PaviaU.mat'),
                 ('sa',  'Salinas_corrected.mat'),
                 ('ksc', 'KSC.mat'),
                 ('botswana', 'Botswana.mat')]
missing = [(d,f) for (d,f) in expected_mats if not os.path.exists(f'{DATA_DIR}/{d}/{f}')]
if missing:
    print('\n!!! Missing datasets — run download cell from sacp_geocp_colab.ipynb first:')
    for d, f in missing: print(f'    {d}/{f}')
else:
    print('\nAll 5 HSI datasets available.')


Mounted at /content/drive
[restore] copied 50 cached pkls from Drive to local
DATA_DIR : /content/drive/MyDrive/sacp_geocp/datasets
CKPT_DIR : /content/hsi_alpha_sweep/checkpoints  (local, primary)
MIRROR   : /content/drive/MyDrive/hsi_alpha_sweep/checkpoints  (Drive, best-effort backup)

All 5 HSI datasets available.


## 3. Helpers — identical to hsi_joint_cv.ipynb

In [3]:
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from scipy.spatial.distance import cdist
from scipy.signal import convolve2d
from scipy import stats as sstats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

class CNN3D(nn.Module):
    def __init__(self, input_channels, n_classes, patch_size=5):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 20, (3,3,3), padding=0)
        self.pool1 = nn.Conv3d(20, 20, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv2 = nn.Conv3d(20, 35, (3,3,3), padding=(1,0,0))
        self.pool2 = nn.Conv3d(35, 35, (3,1,1), stride=(2,1,1), padding=(1,0,0))
        self.conv3 = nn.Conv3d(35, 35, (3,1,1), padding=(1,0,0))
        self.conv4 = nn.Conv3d(35, 35, (2,1,1), stride=(2,1,1), padding=(1,0,0))
        self.patch_size = patch_size; self.input_channels = input_channels
        self.features_size = self._feat_size()
        self.fc = nn.Linear(self.features_size, n_classes)
    def _feat_size(self):
        with torch.no_grad():
            x = torch.zeros((1, 1, self.input_channels, self.patch_size, self.patch_size))
            x = self.pool1(self.conv1(x)); x = self.pool2(self.conv2(x))
            x = self.conv3(x); x = self.conv4(x)
            return int(np.prod(x.size()[1:]))
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool1(x)
        x = F.relu(self.conv2(x)); x = self.pool2(x)
        x = F.relu(self.conv3(x)); x = F.relu(self.conv4(x))
        x = x.view(-1, self.features_size)
        return self.fc(x)

def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def coverage_and_size(psets, labels):
    cov = float(np.mean([int(labels[i]) in s for i, s in enumerate(psets)]))
    sz  = float(np.mean([len(s) for s in psets]))
    return cov, sz

def set_interval_score(psets, true_labels, alpha):
    total = sum(len(psets[i]) + (2.0/alpha) * (0 if int(true_labels[i]) in psets[i] else 1)
                for i in range(len(psets)))
    return float(total / len(psets))

def sscv(psets, y, alpha, buckets=((1,1),(2,2),(3,3),(4,4),(5,10**9))):
    sizes = np.array([len(s) for s in psets])
    covered = np.array([int(y[i]) in psets[i] for i in range(len(y))])
    worst = 0.0; rows = []
    for lo, hi in buckets:
        m = (sizes >= lo) & (sizes <= hi)
        if m.sum() == 0:
            rows.append((lo, hi, 0, None)); continue
        cov = float(covered[m].mean())
        rows.append((lo, hi, int(m.sum()), cov))
        worst = max(worst, abs(cov - (1 - alpha)))
    return float(worst), rows

def sacp_smooth_radius(score_map, H, W, valid_idx, radius, lmd=0.5, k_iter=1):
    N, K = score_map.shape
    if radius <= 0: return score_map.copy()
    mask = np.zeros(N, dtype=bool); mask[valid_idx] = True
    mask_2d = mask.reshape(H, W).astype(np.float64)
    score_2d = score_map.reshape(H, W, K).astype(np.float64)
    k_size = 2 * radius + 1
    kernel = np.ones((k_size, k_size), dtype=np.float64); kernel[radius, radius] = 0.0
    for _ in range(k_iter):
        den = convolve2d(mask_2d, kernel, mode='same', boundary='fill', fillvalue=0.0)
        smoothed = np.empty_like(score_2d)
        for cls in range(K):
            values = np.where(mask_2d > 0, score_2d[..., cls], 0.0)
            num = convolve2d(values, kernel, mode='same', boundary='fill', fillvalue=0.0)
            smoothed[..., cls] = np.where(den > 1e-10, num / np.maximum(den, 1e-10), 0.0)
        new_score = (1 - lmd) * score_2d + lmd * smoothed
        score_2d = np.where(mask_2d[..., None] > 0, new_score, score_2d)
    return score_2d.reshape(N, K)

def vectorised_weighted_quantile(sorted_scores, d_matrix, order, bw, alpha):
    log_w = -0.5 * (d_matrix / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def extract_patches(hsi_chw, indices, patch_size=2):
    d, h, w = hsi_chw.shape
    padded = np.pad(hsi_chw, ((0,0),(patch_size,patch_size),(patch_size,patch_size)), mode='reflect')
    patches = np.zeros((len(indices), 1, d, 2*patch_size+1, 2*patch_size+1), dtype=np.float32)
    for e, idx in enumerate(indices):
        r, c = idx // w, idx % w
        patches[e, 0] = padded[:, r:r+2*patch_size+1, c:c+2*patch_size+1]
    return patches


def set_interval_score_vec(score_mat, q_vec, y_true, alpha):
    """Vectorized IS: score_mat shape (n, K), q_vec shape (n,), y_true shape (n,)."""
    in_set = score_mat < q_vec[:, None]
    sizes = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    penalty = (2.0 / alpha) * (~covered).astype(np.float64)
    return float((sizes + penalty).mean())

DATASET_CONFIG = {
    'ip':       ('ip/Indian_pines_corrected.mat', 'indian_pines_corrected',
                 'ip/Indian_pines_gt.mat', 'indian_pines_gt', 16, 200, 'Indian Pines'),
    'pu':       ('pu/PaviaU.mat', 'paviaU',
                 'pu/PaviaU_gt.mat', 'paviaU_gt', 9, 103, 'Pavia University'),
    'sa':       ('sa/Salinas_corrected.mat', 'salinas_corrected',
                 'sa/Salinas_gt.mat', 'salinas_gt', 16, 204, 'Salinas'),
    'ksc':      ('ksc/KSC.mat', 'KSC',
                 'ksc/KSC_gt.mat', 'KSC_gt', 13, 176, 'KSC'),
    'botswana': ('botswana/Botswana.mat', 'Botswana',
                 'botswana/Botswana_gt.mat', 'Botswana_gt', 14, 145, 'Botswana'),
}
print('Helpers loaded.')


device = cuda
Helpers loaded.


## 4. α-sweep experiment function

Same as `run_joint_cv_experiment` except:
- CV for (r, bw) done at `ALPHA_CV = 0.1` (single α for CV).
- With the CV-selected (r, bw) fixed, generate prediction sets at each α in `ALPHA_GRID = [0.05, 0.10, 0.15]`.
- Save **softmax probs** (`probs_ca`, `probs_te`) + **SACP-smoothed scores per r** (`fcu_per_r`, `ftu_per_r`) so future re-analyses skip retraining.


In [4]:
ALPHA_GRID = (0.05, 0.10, 0.15)
ALPHA_CV   = 0.10

def run_alpha_sweep(data_name, seed, epochs=100, lmd=0.5,
                     radius_grid=(1, 2, 3, 5, 10),
                     bw_grid=(3, 5, 7, 10, 15, 20, 30, 50, 100)):
    ckpt_path = f'{CKPT_DIR}/{data_name}_seed{seed}.pkl'
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'rb') as f: return pickle.load(f)

    torch.manual_seed(seed*100 + 42); np.random.seed(seed*100 + 42)
    rng = np.random.RandomState(seed*100 + 42)

    hf, hk, gf, gk, n_classes, bands, nice = DATASET_CONFIG[data_name]
    hsi = sio.loadmat(f'{DATA_DIR}/{hf}')[hk]
    gt  = sio.loadmat(f'{DATA_DIR}/{gf}')[gk]
    h, w, d = hsi.shape; N = h*w; K = n_classes

    hsi_norm = hsi.astype(np.float32)
    hsi_norm = (hsi_norm - hsi_norm.mean(axis=(0,1))) / (hsi_norm.max() + 1e-8)
    hsi_chw = hsi_norm.transpose(2, 0, 1)

    y_all = gt.reshape(N)
    labeled_idx = np.where(y_all > 0)[0]
    coords = np.column_stack([np.arange(N)//w, np.arange(N)%w]).astype(np.float32)

    rs = seed*100 + 42
    idx_tr, idx_tmp = train_test_split(range(len(labeled_idx)), train_size=250,
                                        stratify=y_all[labeled_idx]-1, random_state=rs)
    idx_ca, idx_te = train_test_split(idx_tmp, test_size=0.5,
                                       stratify=(y_all[labeled_idx]-1)[idx_tmp], random_state=rs)
    tr_gi, ca_gi, te_gi = labeled_idx[idx_tr], labeled_idx[idx_ca], labeled_idx[idx_te]
    y_tr = y_all[tr_gi]-1; y_ca = y_all[ca_gi]-1; y_te = y_all[te_gi]-1

    # ---- Train 3D-CNN ----
    patch_size = 2
    X_tr = extract_patches(hsi_chw, tr_gi, patch_size)
    X_ca = extract_patches(hsi_chw, ca_gi, patch_size)
    X_te = extract_patches(hsi_chw, te_gi, patch_size)

    model = CNN3D(bands, n_classes, patch_size=5).to(device)
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                              batch_size=64, shuffle=True)
    opt = optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()

    model.eval()
    def get_probs(X):
        loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=256, shuffle=False)
        out = []
        with torch.no_grad():
            for (b,) in loader:
                out.append(torch.softmax(model(b.to(device)), dim=1).cpu().numpy())
        return np.concatenate(out)

    probs_ca = get_probs(X_ca); probs_te = get_probs(X_te)
    pred_te = np.argmax(probs_te, axis=1)
    acc = float(np.mean(pred_te == y_te))

    del X_tr, X_ca, X_te, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- APS base scores ----
    cal_all   = aps_scores(probs_ca, rng=rng)
    test_all  = aps_scores(probs_te, rng=rng)
    cal_true  = aps_scores(probs_ca, y_ca, rng=rng)
    coords_ca = coords[ca_gi]; coords_te = coords[te_gi]

    # ---- SACP-smoothed score maps per r (cache full (N,K) maps for val-fold lookup) ----
    sm = np.zeros((N, K))
    for e, i in enumerate(ca_gi): sm[i] = cal_all[e]
    for e, i in enumerate(te_gi): sm[i] = test_all[e]
    valid_idx_all = np.concatenate([ca_gi, te_gi])

    fused_per_r = {r: sacp_smooth_radius(sm, h, w, valid_idx_all, radius=r, lmd=lmd, k_iter=1)
                   for r in radius_grid}
    fcu_per_r = {r: np.array([fused_per_r[r][ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])
                 for r in radius_grid}
    ftu_per_r = {r: np.array([fused_per_r[r][te_gi[e]] for e in range(len(idx_te))])
                 for r in radius_grid}

    # ---- 2D joint CV on cal at ALPHA_CV ----
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_is_sacp  = {r: [] for r in radius_grid}
    cv_is_geocp = {(r, bw): [] for r in radius_grid for bw in bw_grid}
    BATCH_VAL = 500
    for f_tr_idx, f_val_idx in kf.split(np.arange(len(idx_ca))):
        y_cv_val = y_ca[f_val_idx]; n_val = len(f_val_idx)
        for r in radius_grid:
            fcu_tr = fcu_per_r[r][f_tr_idx]
            fcu_val_all = fused_per_r[r][ca_gi[f_val_idx]]
            q_fold = conformal_quantile(fcu_tr, ALPHA_CV)
            q_vec_global = np.full(n_val, q_fold)
            cv_is_sacp[r].append(set_interval_score_vec(fcu_val_all, q_vec_global, y_cv_val, ALPHA_CV))
            order_tr = np.argsort(fcu_tr); sorted_tr = fcu_tr[order_tr]
            q_bw = {bw: np.empty(n_val) for bw in bw_grid}
            for b0 in range(0, n_val, BATCH_VAL):
                b1 = min(b0 + BATCH_VAL, n_val)
                d_batch = cdist(coords_ca[f_val_idx[b0:b1]], coords_ca[f_tr_idx])
                for bw in bw_grid:
                    q_bw[bw][b0:b1] = vectorised_weighted_quantile(sorted_tr, d_batch, order_tr, bw, ALPHA_CV)
                del d_batch
            for bw in bw_grid:
                cv_is_geocp[(r, bw)].append(set_interval_score_vec(fcu_val_all, q_bw[bw], y_cv_val, ALPHA_CV))

    cv_sacp_mean  = {r: float(np.mean(v))  for r, v in cv_is_sacp.items()}
    cv_geocp_mean = {k: float(np.mean(v)) for k, v in cv_is_geocp.items()}
    best_r_sacp = int(min(cv_sacp_mean, key=cv_sacp_mean.get))
    best_rb     = min(cv_geocp_mean, key=cv_geocp_mean.get)
    best_r_geo, best_bw_geo = int(best_rb[0]), int(best_rb[1])

    # ---- Precompute GeoCP weighted quantile at each α in one distance pass ----
    fcu_C = fcu_per_r[best_r_geo]
    ftu_C = ftu_per_r[best_r_geo]
    order_C = np.argsort(fcu_C); sorted_C = fcu_C[order_C]
    BATCH_TEST = 1000; n_te = len(idx_te)
    q_C_per_alpha = {a: np.empty(n_te) for a in ALPHA_GRID}
    for b0 in range(0, n_te, BATCH_TEST):
        b1 = min(b0 + BATCH_TEST, n_te)
        dists = cdist(coords_te[b0:b1], coords_ca)
        for a in ALPHA_GRID:
            q_C_per_alpha[a][b0:b1] = vectorised_weighted_quantile(sorted_C, dists, order_C, best_bw_geo, a)
        del dists

    # ---- α loop: evaluate D/A/B/C at each α (CV (r,bw) fixed at ALPHA_CV) ----
    per_alpha = {}
    for a in ALPHA_GRID:
        q_D = conformal_quantile(cal_true, a)
        ps_D = [np.where(test_all[i] < q_D)[0].tolist() for i in range(n_te)]
        q_A = conformal_quantile(fcu_per_r[1], a)
        ps_A = [np.where(ftu_per_r[1][i] < q_A)[0].tolist() for i in range(n_te)]
        q_B = conformal_quantile(fcu_per_r[best_r_sacp], a)
        ps_B = [np.where(ftu_per_r[best_r_sacp][i] < q_B)[0].tolist() for i in range(n_te)]
        q_C = q_C_per_alpha[a]
        ps_C = [np.where(ftu_C[i] < q_C[i])[0].tolist() for i in range(n_te)]

        row = {}
        for lab, ps in (('D', ps_D), ('A', ps_A), ('B', ps_B), ('C', ps_C)):
            cov, sz = coverage_and_size(ps, y_te)
            is_v    = set_interval_score(ps, y_te, a)
            sscv_v, sscv_rows = sscv(ps, y_te, a)
            row[lab] = {'cov': cov, 'size': sz, 'is': is_v, 'sscv_pct': sscv_v*100,
                        'sscv_per_bucket': sscv_rows, 'pred_sets': ps}
        row['q_C'] = q_C.tolist()
        per_alpha[f'{a:.2f}'] = row

    # ---- Pack + save ----
    result = {
        'dataset': data_name, 'nice_name': nice, 'seed': int(seed),
        'h': int(h), 'w': int(w), 'n_classes': int(n_classes),
        'bands': int(bands), 'lmd': float(lmd),
        'alpha_grid': list(ALPHA_GRID), 'alpha_cv': float(ALPHA_CV),
        'n_train': len(idx_tr), 'n_calib': len(idx_ca), 'n_test': len(idx_te),
        'accuracy': acc,
        'best_r_sacp': best_r_sacp,
        'best_r_geocp': best_r_geo, 'best_bw_geocp': best_bw_geo,
        'per_alpha': per_alpha,
        'cv_is_sacp': cv_sacp_mean,
        'cv_is_geocp': {f'{r}_{bw}': v for (r,bw), v in cv_geocp_mean.items()},
        # raw archives for future re-analysis:
        'probs_ca': probs_ca.astype(np.float32), 'probs_te': probs_te.astype(np.float32),
        'cal_all_aps': cal_all.astype(np.float32), 'test_all_aps': test_all.astype(np.float32),
        'cal_true_aps': cal_true.astype(np.float32),
        'fcu_per_r':  {int(r): v.astype(np.float32) for r, v in fcu_per_r.items()},
        'ftu_per_r':  {int(r): v.astype(np.float32) for r, v in ftu_per_r.items()},
        'ca_gi': ca_gi.astype(np.int64), 'te_gi': te_gi.astype(np.int64),
        'y_ca': y_ca.astype(np.int64), 'y_te': y_te.astype(np.int64),
        'pred_te': pred_te.astype(np.int64),
    }
    tmp = ckpt_path + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path)
    return result

print('run_alpha_sweep defined')


run_alpha_sweep defined


## 5. Run 10 seeds × 5 datasets (resumable)

In [5]:
# --- Defensive: ensure paths exist even if an older cell 4 was run this session ---
LOCAL_DIR      = globals().get('LOCAL_DIR', '/content/hsi_alpha_sweep')
CKPT_DIR       = globals().get('CKPT_DIR', f'{LOCAL_DIR}/checkpoints')
RESULTS_DIR    = globals().get('RESULTS_DIR', f'{LOCAL_DIR}/results')
DRIVE_MIRROR   = globals().get('DRIVE_MIRROR', '/content/drive/MyDrive/hsi_alpha_sweep')
DRIVE_CKPT_DIR = globals().get('DRIVE_CKPT_DIR', f'{DRIVE_MIRROR}/checkpoints')
for _d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)
try: os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
except OSError: pass

DATASETS_TO_RUN = ['ip', 'pu', 'sa', 'ksc', 'botswana']
N_SEEDS = 10
EPOCHS  = 100

log_file = f'{RESULTS_DIR}/run_log.txt'
total_runs = len(DATASETS_TO_RUN) * N_SEEDS
done = 0; t_start = time.time()

def _mirror_to_drive(ds, seed):
    src = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
    dst = f'{DRIVE_CKPT_DIR}/{ds}_seed{seed}.pkl'
    try:
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
        import shutil; shutil.copy2(src, dst)
        return True
    except OSError:
        return False

for ds in DATASETS_TO_RUN:
    print(f'\n{"="*80}\n{DATASET_CONFIG[ds][6]}  ({ds})\n{"="*80}')
    for seed in range(N_SEEDS):
        ckpt = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if os.path.exists(ckpt):
            try:
                with open(ckpt, 'rb') as f: r = pickle.load(f)
            except OSError as e:
                print(f'  seed={seed} cached-read FAILED ({e}); will recompute.')
            else:
                done += 1
                pa = r['per_alpha']
                print(f'  seed={seed} [cached]  acc={r["accuracy"]:.3f}  '
                      f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                      f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                      f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  [{done}/{total_runs}]')
                continue
        t0 = time.time()
        try:
            r = run_alpha_sweep(ds, seed, epochs=EPOCHS)
            done += 1
            mirrored = _mirror_to_drive(ds, seed)
            pa = r['per_alpha']
            msg = (f'  seed={seed}           acc={r["accuracy"]:.3f}  '
                   f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                   f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                   f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  '
                   f'[{time.time()-t0:.0f}s]  [{done}/{total_runs}]  '
                   f'{"mirrored" if mirrored else "(mirror skipped — Drive offline)"}')
            print(msg)
            try:
                with open(log_file, 'a') as f: f.write(f'{ds} {msg}\n')
            except OSError: pass
        except Exception as e:
            import traceback
            print(f'  seed={seed} FAILED: {e}')
            traceback.print_exc()
            try:
                with open(log_file, 'a') as f: f.write(f'{ds} seed={seed} FAILED: {e}\n')
            except OSError: pass

print(f'\n{"="*80}\nALL DONE: {done}/{total_runs} runs  ({(time.time()-t_start)/60:.1f} min)')



Indian Pines  (ip)
  seed=0 [cached]  acc=0.690  α=0.10 IS: D=4.256 A=3.924 B=3.796 C=3.357  (r*=5,bw*=7)  [1/50]
  seed=1 [cached]  acc=0.671  α=0.10 IS: D=4.364 A=3.900 B=3.679 C=3.211  (r*=5,bw*=7)  [2/50]
  seed=2 [cached]  acc=0.662  α=0.10 IS: D=4.561 A=4.058 B=3.902 C=3.453  (r*=5,bw*=3)  [3/50]
  seed=3 [cached]  acc=0.689  α=0.10 IS: D=4.153 A=4.071 B=3.837 C=3.383  (r*=5,bw*=7)  [4/50]
  seed=4 [cached]  acc=0.698  α=0.10 IS: D=4.087 A=3.683 B=3.465 C=3.236  (r*=5,bw*=7)  [5/50]
  seed=5 [cached]  acc=0.706  α=0.10 IS: D=4.374 A=3.835 B=3.723 C=3.350  (r*=5,bw*=10)  [6/50]
  seed=6 [cached]  acc=0.671  α=0.10 IS: D=4.282 A=4.111 B=3.951 C=3.485  (r*=5,bw*=5)  [7/50]
  seed=7 [cached]  acc=0.678  α=0.10 IS: D=4.239 A=3.978 B=3.610 C=3.108  (r*=5,bw*=7)  [8/50]
  seed=8 [cached]  acc=0.651  α=0.10 IS: D=4.889 A=4.095 B=3.841 C=3.461  (r*=5,bw*=7)  [9/50]
  seed=9 [cached]  acc=0.740  α=0.10 IS: D=4.240 A=3.744 B=3.634 C=3.240  (r*=5,bw*=10)  [10/50]

Pavia University  (pu)
  s

## 6. Aggregate → per_seed_alpha.csv + α-sweep summary

In [6]:
rows = []
for ds in DATASETS_TO_RUN:
    for seed in range(N_SEEDS):
        p = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if not os.path.exists(p): continue
        with open(p, 'rb') as f: r = pickle.load(f)
        for a_str, block in r['per_alpha'].items():
            a = float(a_str)
            for lab in ('D', 'A', 'B', 'C'):
                b = block[lab]
                rows.append({
                    'dataset': ds, 'seed': seed, 'alpha': a, 'method': lab,
                    'cov': b['cov'], 'size': b['size'], 'is': b['is'], 'sscv_pct': b['sscv_pct'],
                    'accuracy': r['accuracy'],
                    'r_B': r['best_r_sacp'], 'r_C': r['best_r_geocp'], 'bw_C': r['best_bw_geocp'],
                })

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/per_seed_alpha.csv', index=False)
print('Wrote', f'{RESULTS_DIR}/per_seed_alpha.csv  ({len(df)} rows)')

# Pooled α-sweep summary
summary = (df.groupby(['alpha', 'method'])
             .agg(cov_mean=('cov','mean'), size_mean=('size','mean'),
                  is_mean=('is','mean'), sscv_mean=('sscv_pct','mean'),
                  n=('seed','count')).reset_index())
print('\n=== Pooled α-sweep (n=50 per (α, method)) ===')
print(summary.to_string(index=False))
summary.to_csv(f'{RESULTS_DIR}/alpha_sweep_pooled.csv', index=False)

# Paired (C − A)/A per α
from scipy import stats
print('\n=== C vs A paired improvement per α ===')
for a in sorted(df.alpha.unique()):
    subA = df[(df.alpha == a) & (df.method == 'A')].sort_values(['dataset','seed']).reset_index(drop=True)
    subC = df[(df.alpha == a) & (df.method == 'C')].sort_values(['dataset','seed']).reset_index(drop=True)
    impr = (subA['is'].values - subC['is'].values) / subA['is'].values * 100
    t, p = stats.ttest_1samp(impr, 0.0)
    print(f'  α={a:.2f}: mean improvement = {impr.mean():+.2f}%, t={t:.2f}, p={p:.2e}  (n={len(impr)})')


Wrote /content/hsi_alpha_sweep/results/per_seed_alpha.csv  (600 rows)

=== Pooled α-sweep (n=50 per (α, method)) ===
 alpha method  cov_mean  size_mean  is_mean  sscv_mean  n
  0.05      A  0.950554   1.556140 3.533964   6.718718 50
  0.05      B  0.950696   1.407789 3.379961   6.186654 50
  0.05      C  0.961014   1.431038 2.990489   6.588785 50
  0.05      D  0.949788   1.836703 3.845166   8.143758 50
  0.10      A  0.900082   1.285203 3.283565   9.378536 50
  0.10      B  0.900177   1.207917 3.204387  11.692549 50
  0.10      C  0.912979   1.263122 3.003534  11.501678 50
  0.10      D  0.900448   1.473058 3.464100  12.931641 50
  0.15      A  0.849708   1.142794 3.146692  16.476558 50
  0.15      B  0.849413   1.094192 3.102016  17.240842 50
  0.15      C  0.860220   1.156293 3.020032  17.437852 50
  0.15      D  0.849119   1.274293 3.286041  13.581061 50

=== C vs A paired improvement per α ===
  α=0.05: mean improvement = +13.95%, t=9.96, p=2.33e-13  (n=50)
  α=0.10: mean improvem

## 7. Package & download

In [7]:
import shutil
# 1) Zip the local workspace (has all the pkls + csvs)
zip_local = '/content/hsi_alpha_sweep_results.zip'
shutil.make_archive('/content/hsi_alpha_sweep_results', 'zip', LOCAL_DIR)
print('Zipped to', zip_local)

# 2) Copy zip back to Drive (best-effort)
try:
    os.makedirs(DRIVE_MIRROR, exist_ok=True)
    shutil.copy2(zip_local, f'{DRIVE_MIRROR}/hsi_alpha_sweep_results.zip')
    print('Copied zip to Drive:', f'{DRIVE_MIRROR}/hsi_alpha_sweep_results.zip')
except OSError as e:
    print(f'Drive copy failed ({e}) — download manually via:')
    print('from google.colab import files; files.download("/content/hsi_alpha_sweep_results.zip")')


Zipped to /content/hsi_alpha_sweep_results.zip
Copied zip to Drive: /content/drive/MyDrive/hsi_alpha_sweep/hsi_alpha_sweep_results.zip
